In [1]:
import requests
import pandas as pd
import time

stocks = [
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS",
    "ICICIBANK.NS", "SBIN.NS", "ITC.NS",
    "LT.NS", "HINDUNILVR.NS", "BAJFINANCE.NS"
]
all_data = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for stock in stocks:
    print(f"Scraping {stock}...")
    
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{stock}"
    params = {
        "interval": "1d",
        "range": "1y"
    }
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"❌ Failed {stock}: Status {response.status_code}")
            continue
        
        data = response.json()
        
        if "chart" not in data or data["chart"]["result"] is None:
            print(f"❌ No data for {stock}")
            continue
        
        result = data["chart"]["result"][0]
        
        timestamps = result["timestamp"]
        quotes = result["indicators"]["quote"][0]
        
        df = pd.DataFrame({
            "Date": pd.to_datetime(timestamps, unit='s'),
            "Open": quotes["open"],
            "High": quotes["high"],
            "Low": quotes["low"],
            "Close": quotes["close"],
            "Volume": quotes["volume"]
        })
        
        df["Stock"] = stock
        all_data.append(df)
    
    except Exception as e:
        print(f"❌ Error with {stock}: {e}")
    
    time.sleep(2)  

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    print("✅ Done:", final_df.shape)
    print(final_df.head())
else:
    print("❌ No data collected")

Scraping RELIANCE.NS...
Scraping TCS.NS...
Scraping INFY.NS...
Scraping HDFCBANK.NS...
Scraping ICICIBANK.NS...
Scraping SBIN.NS...
Scraping ITC.NS...
Scraping LT.NS...
Scraping HINDUNILVR.NS...
Scraping BAJFINANCE.NS...
✅ Done: (2490, 7)
                 Date         Open         High          Low        Close  \
0 2025-03-27 03:45:00  1278.150024  1285.000000  1271.300049  1278.199951   
1 2025-03-28 03:45:00  1280.000000  1295.750000  1269.000000  1275.099976   
2 2025-04-01 03:45:00  1264.599976  1277.900024  1249.300049  1252.599976   
3 2025-04-02 03:45:00  1247.550049  1255.550049  1243.900024  1251.150024   
4 2025-04-03 03:45:00  1233.050049  1251.800049  1233.050049  1248.699951   

     Volume        Stock  
0  15028056  RELIANCE.NS  
1  18147129  RELIANCE.NS  
2  12099648  RELIANCE.NS  
3  10142590  RELIANCE.NS  
4   7434366  RELIANCE.NS  


In [4]:
final_df

,Date,Open,High,Low,Close,Volume,Stock
0,2025-03-27 03:45:00,1278.150024,1285.000000,1271.300049,1278.199951,15028056,RELIANCE.NS
1,2025-03-28 03:45:00,1280.000000,1295.750000,1269.000000,1275.099976,18147129,RELIANCE.NS
2,2025-04-01 03:45:00,1264.599976,1277.900024,1249.300049,1252.599976,12099648,RELIANCE.NS
3,2025-04-02 03:45:00,1247.550049,1255.550049,1243.900024,1251.150024,10142590,RELIANCE.NS
4,2025-04-03 03:45:00,1233.050049,1251.800049,1233.050049,1248.699951,7434366,RELIANCE.NS
...,...,...,...,...,...,...,...
2485,2026-03-23 03:45:00,818.099976,818.799988,787.900024,812.599976,15835828,BAJFINANCE.NS
2486,2026-03-24 03:45:00,830.000000,854.000000,817.599976,849.000000,18828248,BAJFINANCE.NS
2487,2026-03-25 03:45:00,862.049988,888.799988,853.299988,882.750000,15995501,BAJFINANCE.NS
2488,2026-03-27 03:45:00,875.000000,875.150024,841.250000,NaN,11282134,BAJFINANCE.NS


In [6]:
final_df.to_csv("final_df.csv")